# DE Zoomcamp 2026 — Module 6: Batch Processing
## Experiment and Homework
- **Dataset:** Yellow Taxi Trip Records from November 2025 ([2025-11 dataset](https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet))
- **Spark UI:** http://localhost:4040 (active while Spark session is running)

---
## 1. Imports & Start Spark Session
Run this first. Spark takes 10–15 seconds to initialize.  
Once this cell finishes, open **http://localhost:4040** in a new tab — the Spark UI will be live.

In [1]:
import os
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("wap_de_zoomcamp_hw6")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"   Spark started successfully")
print(f"   Version  : {spark.version}")
print(f"   UI       : http://localhost:4040")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 02:25:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


   Spark started successfully
   Version  : 4.1.1
   UI       : http://localhost:4040


---
## 2. Read the Parquet File
`count()` is an **action** — it triggers a real Spark job.  
Check the Spark UI Jobs tab after running this cell.

Note: I already downloaded two datasets: yellow_tripdata_2025-11.parquet and taxi_zone_lookup.csv 

In [2]:
df = spark.read.parquet("/app/data/yellow_tripdata_2025-11.parquet")

print(f"Total rows      : {df.count():,}")
print(f"Partitions      : {df.rdd.getNumPartitions()}")
print(f"Columns         : {len(df.columns)}")
print()
df.printSchema()

Total rows      : 4,181,444
Partitions      : 16
Columns         : 20

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



---
## 3. Quick Data Preview & Null Check
Always sanity-check your data before analysis.

In [3]:
# Preview first 5 rows
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [4]:
# Check nulls in key columns
from pyspark.sql.functions import col, count, when

key_cols = ["tpep_pickup_datetime", "tpep_dropoff_datetime",
            "PULocationID", "DOLocationID", "trip_distance", "fare_amount"]

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in key_cols
]).show()

[Stage 6:=====================================================>   (15 + 1) / 16]

+--------------------+---------------------+------------+------------+-------------+-----------+
|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|trip_distance|fare_amount|
+--------------------+---------------------+------------+------------+-------------+-----------+
|                   0|                    0|           0|           0|            0|          0|
+--------------------+---------------------+------------+------------+-------------+-----------+



---
## 4. Homeworks
### Q1 — Spark Version
**Question:** What is the output of `spark.version`?

In [5]:
print(f"Answer Q1 — Spark version: {spark.version}")

Answer Q1 — Spark version: 4.1.1


---
### Q2 — Average Parquet File Size After Repartition
**Question:** Repartition to 4 partitions, save as parquet. What is the average file size?

**Spark UI tip:** After running this cell, go to the **Jobs** tab → click the parquet job →  
you'll see 2 stages: one for the shuffle (repartition) and one for the write.

In [6]:
output_path = "/app/output/yellow_2025_11_4parts"

# Repartition to 4 (triggers shuffle) and write
df.repartition(4).write.parquet(output_path, mode="overwrite")

# Measure file sizes
files = sorted([f for f in os.listdir(output_path) if f.endswith(".parquet")])
sizes = [os.path.getsize(os.path.join(output_path, f)) for f in files]
avg_mb = sum(sizes) / len(sizes) / 1024 / 1024

print("Files written:")
for f, s in zip(files, sizes):
    print(f"  {f[:40]}  →  {s/1024/1024:.1f} MB")

print(f"\nAnswer Q2 — Average file size: {avg_mb:.0f} MB")

Files written:
  part-00000-eb707b75-671a-40d6-8a05-53971  →  24.4 MB
  part-00001-eb707b75-671a-40d6-8a05-53971  →  24.4 MB
  part-00002-eb707b75-671a-40d6-8a05-53971  →  24.4 MB
  part-00003-eb707b75-671a-40d6-8a05-53971  →  24.4 MB

Answer Q2 — Average file size: 24.4 MB


For this question's answer, the rounding up size is 25 MB

---
### Q3 — Trips on November 15
**Question:** How many trips started on November 15, 2025?

We'll run both DataFrame API and Spark SQL side by side — same result, both methods are valid.

In [9]:
# Register as temp view so we can use SQL
df.createOrReplaceTempView("yellow_taxi")

# Method A — DataFrame API
count_api = df.filter(
    F.to_date(F.col("tpep_pickup_datetime")) == "2025-11-15"
).count()

# Method B — Spark SQL
count_sql = spark.sql("""
    SELECT COUNT(*) AS trip_count
    FROM yellow_taxi
    WHERE DATE(tpep_pickup_datetime) = '2025-11-15'
""").collect()[0][0]

print(f"DataFrame API  : {count_api:,}")
print(f"Spark SQL      : {count_sql:,}")
print(f"\nAnswer Q3 — Trips on Nov 15: {count_api:,}")

DataFrame API  : 162,604
Spark SQL      : 162,604

Answer Q3 — Trips on Nov 15: 162,604


In [10]:
# Exploration — Hourly distribution on Nov 15 (not graded, just interesting)
spark.sql("""
    SELECT
        HOUR(tpep_pickup_datetime) AS hour_of_day,
        COUNT(*) AS trips
    FROM yellow_taxi
    WHERE DATE(tpep_pickup_datetime) = '2025-11-15'
    GROUP BY 1
    ORDER BY 1
""").show(24)

[Stage 18:=================================================>      (14 + 2) / 16]

+-----------+-----+
|hour_of_day|trips|
+-----------+-----+
|          0| 8438|
|          1| 6471|
|          2| 4520|
|          3| 3222|
|          4| 1809|
|          5|  906|
|          6| 1447|
|          7| 2091|
|          8| 3269|
|          9| 4812|
|         10| 6163|
|         11| 6976|
|         12| 7867|
|         13| 8286|
|         14| 7910|
|         15| 8800|
|         16| 9256|
|         17| 9952|
|         18|10593|
|         19|10798|
|         20|10488|
|         21| 8356|
|         22| 9950|
|         23|10224|
+-----------+-----+



---
### Q4 — Longest Trip Duration in Hours
**Question:** What is the length of the longest trip in hours?

`unix_timestamp()` converts timestamps to seconds since epoch (integer).  
Subtract → divide by 3600 → duration in hours.

In [11]:
df_dur = df.withColumn(
    "duration_hours",
    (F.unix_timestamp("tpep_dropoff_datetime")
     - F.unix_timestamp("tpep_pickup_datetime")) / 3600
)

max_hours = df_dur.agg(F.max("duration_hours")).collect()[0][0]
print(f"Answer Q4 — Longest trip: {max_hours:.2f} hours")

Answer Q4 — Longest trip: 90.65 hours


In [12]:
# Inspect the top 5 outliers
print("Top 5 longest trips (these are data quality outliers):")
df_dur.orderBy(F.col("duration_hours").desc()) \
    .select("tpep_pickup_datetime", "tpep_dropoff_datetime",
            "trip_distance", "fare_amount", "duration_hours") \
    .limit(5) \
    .show(truncate=False)

Top 5 longest trips (these are data quality outliers):


[Stage 24:==========================================>             (12 + 4) / 16]

+--------------------+---------------------+-------------+-----------+-----------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|duration_hours   |
+--------------------+---------------------+-------------+-----------+-----------------+
|2025-11-26 20:22:12 |2025-11-30 15:01:00  |121.17       |887.1      |90.64666666666666|
|2025-11-27 04:22:41 |2025-11-30 09:19:35  |1.08         |7.9        |76.94833333333334|
|2025-11-03 10:42:55 |2025-11-06 14:55:45  |0.0          |3.0        |76.21388888888889|
|2025-11-07 11:23:22 |2025-11-10 08:40:41  |6.54         |31.0       |69.28861111111111|
|2025-11-18 17:12:47 |2025-11-21 12:17:37  |0.76         |9.3        |67.08055555555555|
+--------------------+---------------------+-------------+-----------+-----------------+



---
### Q5 — Spark UI Port
**Question:** On which local port does the Spark UI run?

In [13]:
print("Answer Q5 — Spark UI port: 4040")
print("Visit http://localhost:4040 while this notebook is running")

Answer Q5 — Spark UI port: 4040
Visit http://localhost:4040 while this notebook is running


---
### Q6 — Least Frequent Pickup Zone
**Question:** Using zone lookup data, what is the LEAST frequent pickup zone?

**Spark UI tip:** After running this cell, go to the **SQL** tab in Spark UI.  
You'll see the full query plan. Look for `BroadcastHashJoin` — Spark auto-detects  
that the zones table is tiny and broadcasts it to avoid a shuffle.

In [14]:
# Load zone lookup
zones = spark.read.csv(
    "/app/data/taxi_zone_lookup.csv",
    header=True,
    inferSchema=True
)
zones.createOrReplaceTempView("zones")

print(f"Zone lookup rows: {zones.count()}")
zones.show(5)

Zone lookup rows: 265
+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [15]:
# Join trips → zones, count by zone, sort ascending
result = spark.sql("""
    SELECT
        z.Zone,
        z.Borough,
        COUNT(*) AS trip_count
    FROM yellow_taxi t
    LEFT JOIN zones z ON t.PULocationID = z.LocationID
    WHERE z.Zone IS NOT NULL
    GROUP BY z.Zone, z.Borough
    ORDER BY trip_count ASC
    LIMIT 10
""")

print("The 10 least frequent pickup zones:")
result.show(truncate=False)

least_zone = result.first()["Zone"]
print(f"\nAnswer Q6 — Least frequent zone: {least_zone}")

The 10 least frequent pickup zones:


+---------------------------------------------+-------------+----------+
|Zone                                         |Borough      |trip_count|
+---------------------------------------------+-------------+----------+
|Governor's Island/Ellis Island/Liberty Island|Manhattan    |1         |
|Arden Heights                                |Staten Island|1         |
|Eltingville/Annadale/Prince's Bay            |Staten Island|1         |
|Port Richmond                                |Staten Island|3         |
|Rossville/Woodrow                            |Staten Island|4         |
|Rikers Island                                |Bronx        |4         |
|Green-Wood Cemetery                          |Brooklyn     |4         |
|Great Kills                                  |Staten Island|4         |
|Jamaica Bay                                  |Queens       |5         |
|Westerleigh                                  |Staten Island|12        |
+---------------------------------------------+----

Actually there are three zones with the same trip count amount of 1: 
1. Governor's Island/Ellis Island/Liberty Island
2. Arden Heights
3. Eltingville/Annadale/Prince's Bay

---
### Final Summary — All Answers

In [16]:
print("=" * 55)
print("  DE ZOOMCAMP 2026 — MODULE 6 HOMEWORK ANSWERS")
print("=" * 55)
print(f"  Q1  Spark Version       :  {spark.version}")
print(f"  Q2  Avg Parquet Size    :  {avg_mb:.1f} MB  →  25MB")
print(f"  Q3  Trips on Nov 15     :  {count_api:,}")
print(f"  Q4  Longest Trip        :  {max_hours:.2f} hours")
print(f"  Q5  Spark UI Port       :  4040")
print(f"  Q6  Least Frequent Zone :  {least_zone}")
print("=" * 55)

  DE ZOOMCAMP 2026 — MODULE 6 HOMEWORK ANSWERS
  Q1  Spark Version       :  4.1.1
  Q2  Avg Parquet Size    :  24.4 MB  →  25MB
  Q3  Trips on Nov 15     :  162,604
  Q4  Longest Trip        :  90.65 hours
  Q5  Spark UI Port       :  4040
  Q6  Least Frequent Zone :  Governor's Island/Ellis Island/Liberty Island


## Further Data Exploration
It's Q4 further exploration: Where Do the Top 5 Longest Trips Start and End?

In [17]:
# Create two separate aliases of the zones table
# Needed because we join the same table twice
pu_zones = zones.select(
    F.col("LocationID").alias("pu_location_id"),
    F.col("Zone").alias("pickup_zone"),
    F.col("Borough").alias("pickup_borough")
)

do_zones = zones.select(
    F.col("LocationID").alias("do_location_id"),
    F.col("Zone").alias("dropoff_zone"),
    F.col("Borough").alias("dropoff_borough")
)

# Join trips → pickup zones → dropoff zones
# Both use broadcast() since zones is tiny (~265 rows)
top5 = (
    df_dur
    .orderBy(F.col("duration_hours").desc())
    .limit(5)
    .join(broadcast(pu_zones),
          df_dur.PULocationID == pu_zones.pu_location_id, "left")
    .join(broadcast(do_zones),
          df_dur.DOLocationID == do_zones.do_location_id, "left")
    .select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        F.round("duration_hours", 2).alias("duration_hours"),
        "trip_distance",
        "fare_amount",
        "pickup_borough",
        "pickup_zone",
        "dropoff_borough",
        "dropoff_zone",
    )
)

top5.show(truncate=False)

[Stage 40:==========================================>             (12 + 4) / 16]

+--------------------+---------------------+--------------+-------------+-----------+--------------+--------------------------------+---------------+----------------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|duration_hours|trip_distance|fare_amount|pickup_borough|pickup_zone                     |dropoff_borough|dropoff_zone                |
+--------------------+---------------------+--------------+-------------+-----------+--------------+--------------------------------+---------------+----------------------------+
|2025-11-26 20:22:12 |2025-11-30 15:01:00  |90.65         |121.17       |887.1      |N/A           |Outside of NYC                  |N/A            |Outside of NYC              |
|2025-11-27 04:22:41 |2025-11-30 09:19:35  |76.95         |1.08         |7.9        |Manhattan     |West Chelsea/Hudson Yards       |Manhattan      |Clinton East                |
|2025-11-03 10:42:55 |2025-11-06 14:55:45  |76.21         |0.0          |3.0        |Queens        |Saint

In [18]:
print("Top 5 Longest Trips — Route Summary")
print("=" * 70)

rows = top5.collect()
for i, row in enumerate(rows, 1):
    pu_zone = row.pickup_zone  or "Unknown"
    do_zone = row.dropoff_zone or "Unknown"
    pu_boro = row.pickup_borough  or "?"
    do_boro = row.dropoff_borough or "?"

    print(f"\n  Trip #{i}")
    print(f"  Duration   : {row.duration_hours} hours")
    print(f"  Distance   : {row.trip_distance} miles")
    print(f"  Fare       : ${row.fare_amount}")
    print(f"  Pickup     : {pu_zone} ({pu_boro})")
    print(f"  Dropoff    : {do_zone} ({do_boro})")
    print(f"  Started    : {row.tpep_pickup_datetime}")
    print(f"  Ended      : {row.tpep_dropoff_datetime}")

Top 5 Longest Trips — Route Summary


[Stage 43:==========================================>             (12 + 4) / 16]


  Trip #1
  Duration   : 90.65 hours
  Distance   : 121.17 miles
  Fare       : $887.1
  Pickup     : Outside of NYC (N/A)
  Dropoff    : Outside of NYC (N/A)
  Started    : 2025-11-26 20:22:12
  Ended      : 2025-11-30 15:01:00

  Trip #2
  Duration   : 76.95 hours
  Distance   : 1.08 miles
  Fare       : $7.9
  Pickup     : West Chelsea/Hudson Yards (Manhattan)
  Dropoff    : Clinton East (Manhattan)
  Started    : 2025-11-27 04:22:41
  Ended      : 2025-11-30 09:19:35

  Trip #3
  Duration   : 76.21 hours
  Distance   : 0.0 miles
  Fare       : $3.0
  Pickup     : Saint Michaels Cemetery/Woodside (Queens)
  Dropoff    : N/A (Unknown)
  Started    : 2025-11-03 10:42:55
  Ended      : 2025-11-06 14:55:45

  Trip #4
  Duration   : 69.29 hours
  Distance   : 6.54 miles
  Fare       : $31.0
  Pickup     : JFK Airport (Queens)
  Dropoff    : West Chelsea/Hudson Yards (Manhattan)
  Started    : 2025-11-07 11:23:22
  Ended      : 2025-11-10 08:40:41

  Trip #5
  Duration   : 67.08 hours
  

In [19]:
# A common data quality pattern: meter left running, never properly ended
# These trips show pickup zone == dropoff zone despite 90+ hour duration
print("Same zone pickup and dropoff? (meter left running?)")
print()

for i, row in enumerate(rows, 1):
    same = row.pickup_zone == row.dropoff_zone
    flag = "⚠️  SAME ZONE" if same else "✅ Different zones"
    print(f"  Trip #{i}  {flag}")
    print(f"           {row.pickup_zone}  →  {row.dropoff_zone}")

Same zone pickup and dropoff? (meter left running?)

  Trip #1  ⚠️  SAME ZONE
           Outside of NYC  →  Outside of NYC
  Trip #2  ✅ Different zones
           West Chelsea/Hudson Yards  →  Clinton East
  Trip #3  ✅ Different zones
           Saint Michaels Cemetery/Woodside  →  N/A
  Trip #4  ✅ Different zones
           JFK Airport  →  West Chelsea/Hudson Yards
  Trip #5  ⚠️  SAME ZONE
           Penn Station/Madison Sq West  →  Penn Station/Madison Sq West


What You'll Find in the Results
The last cell is the most revealing. These outlier trips almost always follow
one of two patterns:
1. Same zone, 90+ hours: The taxi meter was never properly closed. The driver forgot to end the trip in the system after the passenger left — meter kept running for days.
2. Different zones, impossible fare: A system clock error — the dropoff timestamp was either error or entered incorrectly, making a normal trip appear to last days.

### TIPS
Always add a sanity filter on your production data pipelines before any analysis

In [25]:
# Filter out outliers before real analysis
df_clean = df_dur.filter(
    (F.col("duration_hours") > 0) &
    (F.col("duration_hours") < 24)
)

print(f"Total rows before sanity filter (Raw Data)    : {df_dur.count():,}")
print(f"Total rows after sanity filter (Clean Data)   : {df_clean.count():,}")
print(f"Clean Data Proportion                         : {(df_clean.count()/df_dur.count())*100:.2f}%")

Total rows before sanity filter (Raw Data)    : 4,181,444
Total rows after sanity filter (Clean Data)   : 4,119,282
Clean Data Proportion                         : 98.51%


---
## Stop Spark — Always Run This Last Cell

In [26]:
spark.stop()
print("✅ Spark session stopped. Memory released.")

✅ Spark session stopped. Memory released.
